# Nigeria Immunisation Modelling — Domains 1, 2 & 5
## NPHCDA Digital Innovation Hub | UNICEF Technical Assistance

**Principal Investigator:** Dr. Amobi Andrew Onovo, PhD, MPH  
**Consortium:** CIDRE–Quantium Insights | March 2026

| Domain | Question | Method |
|--------|----------|--------|
| **1** | Which antigens are projected to fall below 80% in the next 6–12 months? | Prophet time-series + 80%/95% prediction intervals |
| **2** | What drives Penta1→Penta3, Penta1→Measles1 and Measles1→Measles2 dropout? | Prophet + LASSO feature selection |
| **5-II** | What community archetypes explain persistent zero-dose? | Hierarchical Agglomerative Clustering (Ward, k=5) |

**Data sources:** DHIS2 (Jan 2021 – Dec 2025) | NDHS 2024 predictor dataset | 37 states



---
## Section 0 — Environment Setup

In [1]:
try:
    import google.colab
    IN_COLAB = True
    print("Environment: Google Colab")
except ImportError:
    IN_COLAB = False
    print("Environment: Local Jupyter")


Environment: Google Colab


In [2]:
import subprocess, sys

packages = [
    "prophet", "scikit-learn", "scipy", "seaborn",
    "pandas", "numpy", "matplotlib",
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("\n✅ Packages installed.")



✅ Packages installed.


---
## Section 1 — Upload Data Files

Upload both files when the picker appears:
- `dhis2_data_all_states.csv`
- `nigeria_zero_dose_model_dataset.csv`

In [3]:
if IN_COLAB:
    from google.colab import files
    print("Select dhis2_data_all_states.csv and nigeria_zero_dose_model_dataset.csv")
    uploaded = files.upload()
    for f in uploaded:
        print(f"✅ Uploaded: {f}  ({len(uploaded[f])/1024:.0f} KB)")
else:
    import os
    for f in ["dhis2_data_all_states.csv","nigeria_zero_dose_model_dataset.csv"]:
        print(f"  {'✅' if os.path.exists(f) else '❌ MISSING'}  {f}")


Select dhis2_data_all_states.csv and nigeria_zero_dose_model_dataset.csv


Saving dhis2_data_all_states.csv to dhis2_data_all_states.csv
Saving nigeria_zero_dose_model_dataset.csv to nigeria_zero_dose_model_dataset.csv
✅ Uploaded: dhis2_data_all_states.csv  (5453 KB)
✅ Uploaded: nigeria_zero_dose_model_dataset.csv  (7 KB)


---
## Section 2 — Imports and Configuration

In [4]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # change to "inline" for Jupyter display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist
from scipy.stats import kruskal
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LassoCV
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from prophet import Prophet

warnings.filterwarnings("ignore")
np.random.seed(42)

DHIS2_PATH      = "dhis2_data_all_states.csv"
MODEL_PATH      = "nigeria_zero_dose_model_dataset.csv"
OUTPUT_DIR      = "outputs"
FORECAST_MONTHS = 18
THRESHOLD       = 80.0
N_CLUSTERS      = 5
os.makedirs(OUTPUT_DIR, exist_ok=True)

ZONE_PAL = {
    "North West":"#D73027","North East":"#F46D43","North Central":"#FDAE61",
    "South West":"#1A9850","South East":"#91CF60","South South":"#66C2A5",
}
ANTIGEN_PAL = {
    "BCG":"#1565C0","Penta1":"#2E7D32","Penta3":"#E65100","Measles1":"#880E4F",
}
ARCHETYPE_COLORS_NAMED = {
    "High-Burden Remote Rural":         "#D73027",
    "Conflict-Affected / Hard-to-Reach":"#F46D43",
    "Nomadic / High-Mobility":          "#880E4F",
    "Urban Underserved":                "#1565C0",
    "Moderate Access — Improving":      "#2E7D32",
}
BG = "#FAFBFC"
print("Configuration ready.")


Configuration ready.


---
## Section 3 — Load and Prepare DHIS2 Data

In [5]:
d = pd.read_csv(DHIS2_PATH)
count_cols = ["bcg_count","penta_1_count","pent_2_count","penta_3_count",
              "measles_1_count","measles_2_count"]
for c in count_cols:
    d[c] = pd.to_numeric(d[c], errors="coerce")
d["ds"]    = pd.to_datetime(d["period"], format="%b-%y", errors="coerce")
d["year"]  = d["ds"].dt.year
d["month"] = d["ds"].dt.month

agg = (d.groupby(["state","zone","year","month","ds"])[count_cols]
        .sum().reset_index().sort_values(["state","ds"]).reset_index(drop=True))
agg["dropout_p1p3"] = ((agg["penta_1_count"] - agg["penta_3_count"])
                        / agg["penta_1_count"] * 100).clip(-50, 100)
agg["dropout_p1m1"] = ((agg["penta_1_count"] - agg["measles_1_count"])
                        / agg["penta_1_count"] * 100).clip(-50, 100)
agg["dropout_m1m2"] = ((agg["measles_1_count"] - agg["measles_2_count"])
                        / agg["measles_1_count"] * 100).clip(-50, 100)

nat = agg.groupby("ds")[["penta_1_count","penta_3_count","measles_1_count",
                          "measles_2_count","bcg_count"]].sum().reset_index()
nat["dropout_p1p3"] = ((nat["penta_1_count"]-nat["penta_3_count"])/nat["penta_1_count"]*100).clip(-50,100)
nat["dropout_p1m1"] = ((nat["penta_1_count"]-nat["measles_1_count"])/nat["penta_1_count"]*100).clip(-50,100)
nat["dropout_m1m2"] = ((nat["measles_1_count"]-nat["measles_2_count"])/nat["measles_1_count"]*100).clip(-50,100)

print(f"States: {agg['state'].nunique()} | Period: {agg['ds'].min().date()} – {agg['ds'].max().date()}")
print(f"National monthly rows: {len(nat)}")
agg.head(3)


States: 37 | Period: 2021-01-01 – 2025-12-01
National monthly rows: 60


,state,zone,year,month,ds,bcg_count,penta_1_count,pent_2_count,penta_3_count,measles_1_count,measles_2_count,dropout_p1p3,dropout_p1m1,dropout_m1m2
0,Abia,South East,2021,1,2021-01-01,7125.0,7211.0,5029.0,5993.0,6375.0,2233.0,16.890861,11.593399,64.972549
1,Abia,South East,2021,2,2021-02-01,7154.0,7921.0,6524.0,7447.0,6927.0,3096.0,5.984093,12.548921,55.305327
2,Abia,South East,2021,3,2021-03-01,6381.0,8320.0,6672.0,8220.0,7085.0,3180.0,1.201923,14.843750,55.116443


### 3.1 Hybrid imputation of missing state-month dose counts

Several states (Katsina, Kogi, Kwara, Lagos, Niger) joined DHIS2 reporting late, producing
empty 2021–2022 cells in the downstream dropout heatmaps. The cell below imputes the missing
monthly dose counts using an equal-weight ensemble of two methods — a KNN imputer and an
iterative BayesianRidge regressor (MICE-style) — followed by a Predictive Mean Matching
(PMM) snap step that replaces each ensemble prediction with the nearest observed value from
the same zone and same month-of-year. The output is a new `agg_imputed` DataFrame plus an
`is_imputed` boolean flag column. Downstream cells read `agg_imputed` instead of `agg`.

**Validation.** A 10% hold-out fold reports the MAE per count column at the bottom of the
cell, giving reviewers a calibration number for each antigen.


In [6]:
# Cell 9b — Hybrid PMM + KNN imputation of missing state-month dose counts
# Why this cell exists: several states (notably Katsina, Kogi, Kwara, Lagos,
# Niger) joined DHIS2 reporting late, producing empty 2021–2022 cells in the
# downstream dropout heatmaps. We impute the missing monthly dose counts —
# never the dropout rate directly — so the count relationships that feed
# every downstream metric stay internally consistent.
#
# Algorithm (equal-weight ensemble, average-then-snap PMM):
#   1. Build a feature matrix per (state × ds): the six dose counts, year,
#      month, and one-hot zone.
#   2. Run KNN imputer (k=5, weighted by distance) → fill #1
#   3. Run IterativeImputer with BayesianRidge → fill #2
#   4. Average #1 and #2 → continuous predictions
#   5. PMM snap: for each imputed (state, ds, antigen), replace the prediction
#      with the nearest observed value from the donor pool of (same zone,
#      same month-of-year, antigen-specific). This is the "predictive mean
#      matching" property — every imputed value is a real observed value
#      from a comparable context, never synthetic.
#   6. Recompute dropout columns from the imputed counts.
#
# Output: `agg_imputed` (same shape as `agg`) plus `is_imputed` boolean
# flag column. Downstream cells read `agg_imputed` instead of `agg`.

import warnings
warnings.filterwarnings("ignore")
from sklearn.experimental import enable_iterative_imputer  # noqa: F401, must precede IterativeImputer import
from sklearn.impute import KNNImputer, IterativeImputer
from sklearn.linear_model import BayesianRidge

COUNT_COLS = ["bcg_count", "penta_1_count", "pent_2_count", "penta_3_count",
              "measles_1_count", "measles_2_count"]

# === 0. Build a complete state × month grid so missing rows become explicit NaN
all_states  = sorted(agg["state"].unique())
all_months  = pd.date_range(agg["ds"].min(), agg["ds"].max(), freq="MS")
grid = pd.MultiIndex.from_product([all_states, all_months],
                                    names=["state", "ds"]).to_frame(index=False)
zone_map = agg.drop_duplicates("state").set_index("state")["zone"].to_dict()
grid["zone"]  = grid["state"].map(zone_map)
grid["year"]  = grid["ds"].dt.year
grid["month"] = grid["ds"].dt.month

agg_full = grid.merge(
    agg[["state", "ds"] + COUNT_COLS],
    on=["state", "ds"], how="left")

# Track which cells are missing BEFORE imputation, per count column
missing_mask = agg_full[COUNT_COLS].isna()
n_missing_per_col = missing_mask.sum()
print("Missing cells before imputation (per count column):")
print(n_missing_per_col.to_string())
print(f"Total missing count cells: {int(missing_mask.values.sum()):,}")

# === 1. Feature matrix
X_num    = agg_full[COUNT_COLS + ["year", "month"]].copy()
X_zone   = pd.get_dummies(agg_full["zone"], prefix="zone")
X_full   = pd.concat([X_num, X_zone], axis=1)

# === 2. KNN imputer
knn = KNNImputer(n_neighbors=5, weights="distance")
X_knn = pd.DataFrame(knn.fit_transform(X_full),
                      columns=X_full.columns, index=X_full.index)

# === 3. IterativeImputer with BayesianRidge (MICE-style)
ii = IterativeImputer(estimator=BayesianRidge(),
                       max_iter=15, random_state=7,
                       initial_strategy="median", sample_posterior=False)
X_ii  = pd.DataFrame(ii.fit_transform(X_full),
                      columns=X_full.columns, index=X_full.index)

# === 4. Equal-weight ensemble average
X_ens = 0.5 * X_knn[COUNT_COLS] + 0.5 * X_ii[COUNT_COLS]

# === 5. PMM snap: replace each imputed value with the nearest *observed*
#       value from the donor pool (same zone, same month-of-year, same antigen).
agg_full[COUNT_COLS] = agg_full[COUNT_COLS]   # alias for clarity below

# Build donor pools once
donor_pools = {}
for col in COUNT_COLS:
    obs = agg_full[[col, "zone", "month"]].dropna(subset=[col])
    for (zone, month), grp in obs.groupby(["zone", "month"]):
        donor_pools[(col, zone, month)] = grp[col].values

def pmm_snap(col, zone, month, predicted):
    """Replace `predicted` with the closest observed value in the donor pool."""
    pool = donor_pools.get((col, zone, month))
    if pool is None or len(pool) == 0:
        # Fallback: any observed value of this antigen, regardless of zone/month
        pool = agg_full[col].dropna().values
        if len(pool) == 0:
            return predicted
    idx = np.argmin(np.abs(pool - predicted))
    return float(pool[idx])

# Apply ensemble + PMM only where the original cell was missing
agg_imputed = agg_full.copy()
for col in COUNT_COLS:
    mask = missing_mask[col].values
    if not mask.any():
        continue
    rows = np.where(mask)[0]
    for r in rows:
        pred = X_ens.iloc[r][col]
        z    = agg_imputed.iloc[r]["zone"]
        mo   = int(agg_imputed.iloc[r]["month"])
        snapped = pmm_snap(col, z, mo, pred)
        # Counts can't be negative; floor at zero
        agg_imputed.loc[agg_imputed.index[r], col] = max(snapped, 0.0)

# === 6. Recompute dropout columns from the imputed counts
agg_imputed["dropout_p1p3"] = ((agg_imputed["penta_1_count"] - agg_imputed["penta_3_count"])
                                / agg_imputed["penta_1_count"] * 100).clip(-50, 100)
agg_imputed["dropout_p1m1"] = ((agg_imputed["penta_1_count"] - agg_imputed["measles_1_count"])
                                / agg_imputed["penta_1_count"] * 100).clip(-50, 100)
agg_imputed["dropout_m1m2"] = ((agg_imputed["measles_1_count"] - agg_imputed["measles_2_count"])
                                / agg_imputed["measles_1_count"] * 100).clip(-50, 100)

# Add `is_imputed` flag: True if ANY of the count columns for that row was imputed
agg_imputed["is_imputed"] = missing_mask.any(axis=1).values

# === Validation: hold-out MAE
# Hold out 10% of observed values, re-impute with the same ensemble, report
# mean absolute error — gives reviewers a calibration number for the method.
rng = np.random.default_rng(7)
val_results = {}
for col in COUNT_COLS:
    observed_idx = agg_full.index[~missing_mask[col]].tolist()
    if len(observed_idx) < 50:
        continue
    n_hold = max(1, int(len(observed_idx) * 0.10))
    hold = rng.choice(observed_idx, size=n_hold, replace=False)
    truth = agg_full.loc[hold, col].values.copy()
    Xv = X_full.copy()
    Xv.loc[hold, col] = np.nan
    # Mini ensemble on the validation slice
    Xv_knn = pd.DataFrame(KNNImputer(n_neighbors=5, weights="distance")
                            .fit_transform(Xv),
                          columns=Xv.columns, index=Xv.index)
    Xv_ii  = pd.DataFrame(IterativeImputer(estimator=BayesianRidge(),
                                              max_iter=10, random_state=7,
                                              initial_strategy="median")
                            .fit_transform(Xv),
                          columns=Xv.columns, index=Xv.index)
    pred_ens = 0.5 * Xv_knn.loc[hold, col].values + 0.5 * Xv_ii.loc[hold, col].values
    mae      = float(np.mean(np.abs(pred_ens - truth)))
    rel_mae  = mae / max(np.mean(np.abs(truth)), 1e-6) * 100
    val_results[col] = (mae, rel_mae)

print()
print("Validation — 10% hold-out MAE per count column:")
print(f"{'column':<20} {'MAE':>10} {'rel-MAE (%)':>14}")
for col, (mae, rel) in val_results.items():
    print(f"{col:<20} {mae:>10.1f} {rel:>14.1f}")
print()
print(f"Imputed rows  : {int(agg_imputed['is_imputed'].sum()):,} of {len(agg_imputed):,} "
       f"({agg_imputed['is_imputed'].mean()*100:.1f}%)")
print(f"Output frame  : agg_imputed  ·  flag column: is_imputed")


Missing cells before imputation (per count column):
bcg_count          120
penta_1_count      120
pent_2_count       120
penta_3_count      120
measles_1_count    120
measles_2_count    120
Total missing count cells: 720

Validation — 10% hold-out MAE per count column:
column                      MAE    rel-MAE (%)
bcg_count                1081.8           11.1
penta_1_count             692.4            7.0
pent_2_count              822.2            8.2
penta_3_count             588.1            6.3
measles_1_count          1069.9           10.4
measles_2_count          1909.8           24.5

Imputed rows  : 120 of 2,220 (5.4%)
Output frame  : agg_imputed  ·  flag column: is_imputed


---
## Section 4 — Exploratory Data Analysis

### 4.1 National antigen counts time series

In [7]:
antigen_ts = {"BCG":"bcg_count","Penta1":"penta_1_count",
              "Penta3":"penta_3_count","Measles1":"measles_1_count"}

fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor=BG)
axes = axes.flatten()
for idx, (label, col) in enumerate(antigen_ts.items()):
    ax = axes[idx]; ax.set_facecolor(BG)
    ts = nat[["ds",col]].dropna()
    ax.fill_between(ts["ds"], ts[col]/1e3, alpha=0.22, color=ANTIGEN_PAL[label])
    ax.plot(ts["ds"], ts[col]/1e3, color=ANTIGEN_PAL[label], lw=2.2)
    ax.plot(ts["ds"], ts[col].rolling(3,center=True).mean()/1e3,
            "--", color=ANTIGEN_PAL[label], lw=1.4, alpha=0.7, label="3-month MA")
    ax.set_title(f"{label} — Monthly Doses", fontsize=11, fontweight="bold")
    ax.set_ylabel("Doses (thousands)", fontsize=9); ax.legend(fontsize=8)
    ax.grid(alpha=0.18, ls="--")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:.0f}k"))
fig.suptitle("EDA — National Monthly Antigen Dose Counts (2021–2025)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig_eda_antigen_counts.png", dpi=150, bbox_inches="tight")
plt.show(); print("Figure saved.")


Figure saved.


### 4.2 Penta1 state × month heatmap (2024)

In [8]:
fig, ax = plt.subplots(figsize=(14, 9), facecolor=BG)
piv = (agg[agg["year"]==2024].groupby(["state","month"])["penta_1_count"]
       .mean().unstack(fill_value=np.nan))
piv.columns = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
sns.heatmap(piv, ax=ax, cmap="YlOrRd", linewidths=0.4, linecolor="#ddd",
            cbar_kws={"label":"Penta1 Doses (mean count)"}, annot=False)
ax.set_title("EDA — State × Month Penta1 Counts (2024)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig_eda_penta1_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


### 4.3 Dropout rate distribution by zone (box plots)

In [9]:
zones_order = ["North West","North East","North Central","South West","South East","South South"]
agg_2024 = agg[agg["year"]==2024].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=BG)
for ax, (drop_col, label) in zip(axes, [
    ("dropout_p1p3","Penta1 → Penta3 Dropout Rate (%)"),
    ("dropout_p1m1","Penta1 → Measles1 Dropout Rate (%)"),
]):
    data_by_zone = [agg_2024[agg_2024["zone"]==z][drop_col].dropna().values for z in zones_order]
    bp = ax.boxplot(data_by_zone, patch_artist=True, notch=False,
                    medianprops=dict(color="black",lw=2.2))
    for patch, zone in zip(bp["boxes"], zones_order):
        patch.set_facecolor(ZONE_PAL[zone]); patch.set_alpha(0.78)
    ax.axhline(0, color="#C62828", ls="--", lw=1.5, alpha=0.7)
    ax.set_xticklabels([z.replace(" ","\n") for z in zones_order], fontsize=8.5)
    ax.set_ylabel(label, fontsize=10); ax.set_title(label, fontsize=10.5, fontweight="bold")
    ax.grid(axis="y", alpha=0.2, ls="--"); ax.set_facecolor(BG)
fig.suptitle("EDA Domain 2 — Dropout Rate Distribution by Zone (2024)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig_eda_dropout_boxplot_zone.png", dpi=150, bbox_inches="tight")
plt.show()


---
## Section 5 — Domain 1: Antigen Coverage Forecasting

**Question:** Which routine immunisation antigens are projected to fall below the
80% national coverage target in the next 6–12 months — at national, **LGA**, and
**ward** levels?

**Method:** Prophet (Facebook/Meta) time-series with yearly + semi-annual seasonality,
fit per antigen at each spatial level. Forecasts are expressed as % of the 2024
baseline so they are bounded and directly comparable to the 80% national target.
Lower 80% and 95% prediction intervals are retained for uncertainty.

**What the forecast plot shows.** A dashed red line marks the 80% national target.
Months where the central forecast falls below the line are flagged as
*at-risk* and shaded. A panel headline lists antigens that cross 80% inside the
6–12-month window.

**Spatial scaling.** National → LGA forecasts are produced from the same Prophet
configuration. Ward-level forecasts require ward-tagged DHIS2 extracts (the same
pipeline applies; see Section 5.3).


In [10]:
def run_prophet(ts_df, antigen, periods=FORECAST_MONTHS):
    df = ts_df[["ds","y"]].dropna().copy()
    m  = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                 daily_seasonality=False, interval_width=0.95,
                 changepoint_prior_scale=0.05, seasonality_prior_scale=10)
    m.add_seasonality(name="semi_annual", period=182.5, fourier_order=3)
    m.fit(df)
    future = m.make_future_dataframe(periods=periods, freq="MS")
    fc     = m.predict(future)
    fc["antigen"] = antigen
    return fc, m

all_national_fc = []
print("Fitting Prophet models...")
for antigen, count_col in antigen_ts.items():
    ts = nat[["ds", count_col]].rename(columns={"ds":"ds", count_col:"y"})
    fc, _ = run_prophet(ts, antigen)
    all_national_fc.append(fc)
    last = fc[fc["ds"] > nat["ds"].max()].iloc[-1]
    print(f"  {antigen}: Dec 2026 forecast = {last['yhat']/1e3:.0f}k "
          f"(95% PI: {last['yhat_lower']/1e3:.0f}k – {last['yhat_upper']/1e3:.0f}k)")


Fitting Prophet models...
  BCG: Dec 2026 forecast = 348k (95% PI: 292k – 399k)
  Penta1: Dec 2026 forecast = 331k (95% PI: 260k – 397k)
  Penta3: Dec 2026 forecast = 333k (95% PI: 267k – 398k)
  Measles1: Dec 2026 forecast = 353k (95% PI: 296k – 414k)


In [11]:
# Cell 18 — National forecast: % of 2024 baseline, with explicit 80% threshold
# Re-expresses each Prophet forecast as % of the 2024 mean monthly dose volume so
# the 80% coverage target becomes a meaningful horizontal line, and antigens that
# fall below it inside the 6–12 month window can be identified visually.

THRESHOLD_PCT = 80.0
WINDOW_MONTHS = (6, 12)  # at-risk window (months ahead)

fig, axes = plt.subplots(2, 2, figsize=(18, 13), facecolor=BG)
axes = axes.flatten()

at_risk_summary = []   # collected for headline table below

for idx, (antigen, count_col) in enumerate(antigen_ts.items()):
    fc       = all_national_fc[idx]
    cutoff   = nat["ds"].max()
    ts_obs   = nat[["ds", count_col]].dropna()
    base_2024 = ts_obs[ts_obs["ds"].dt.year == 2024][count_col].mean()

    # Convert every series to % of 2024 baseline
    fc_pct = fc.copy()
    fc_pct["yhat"]       = fc["yhat"]       / base_2024 * 100
    fc_pct["yhat_lower"] = fc["yhat_lower"] / base_2024 * 100
    fc_pct["yhat_upper"] = fc["yhat_upper"] / base_2024 * 100
    obs_pct = ts_obs[count_col] / base_2024 * 100

    hist_fc = fc_pct[fc_pct["ds"] <= cutoff]
    fore_fc = fc_pct[fc_pct["ds"] >  cutoff].reset_index(drop=True)
    fore_fc["months_ahead"] = ((fore_fc["ds"].dt.year - cutoff.year) * 12
                              + (fore_fc["ds"].dt.month - cutoff.month))
    window_fc = fore_fc[(fore_fc["months_ahead"] >= WINDOW_MONTHS[0])
                       & (fore_fc["months_ahead"] <= WINDOW_MONTHS[1])]

    ax = axes[idx]; ax.set_facecolor(BG)

    # Observed
    ax.scatter(ts_obs["ds"], obs_pct, s=14, alpha=0.65,
               color=ANTIGEN_PAL[antigen], zorder=5, label="Observed")
    # Fitted history
    ax.plot(hist_fc["ds"], hist_fc["yhat"], lw=2.2,
            color=ANTIGEN_PAL[antigen], alpha=0.9, label="Fitted")
    # Forecast central + intervals
    ax.plot(fore_fc["ds"], fore_fc["yhat"], "--", lw=2.4, color=ANTIGEN_PAL[antigen],
            label="Forecast")
    ax.fill_between(fore_fc["ds"], fore_fc["yhat_lower"], fore_fc["yhat_upper"],
                    alpha=0.13, color=ANTIGEN_PAL[antigen], label="95% PI")
    lo80 = fore_fc["yhat"] - (fore_fc["yhat"] - fore_fc["yhat_lower"]) * 0.53
    hi80 = fore_fc["yhat"] + (fore_fc["yhat_upper"] - fore_fc["yhat"]) * 0.53
    ax.fill_between(fore_fc["ds"], lo80, hi80, alpha=0.25,
                    color=ANTIGEN_PAL[antigen], label="80% PI")

    # ── 80% threshold line + below-threshold shading ──────────────────────
    ax.axhline(THRESHOLD_PCT, color="#C62828", ls="--", lw=1.8, alpha=0.85,
               label=f"{THRESHOLD_PCT:.0f}% target")
    ymin, _ = ax.get_ylim()
    ax.axhspan(0, THRESHOLD_PCT, color="#C62828", alpha=0.05, zorder=1)

    # Mark each forecast month that falls below the threshold
    below = fore_fc[fore_fc["yhat"] < THRESHOLD_PCT]
    if len(below):
        ax.scatter(below["ds"], below["yhat"], color="#C62828", zorder=7,
                   s=58, marker="v", edgecolor="white", linewidth=0.6,
                   label="Below 80%")

    # Forecast-period cutoff
    ax.axvline(cutoff, color="#555", ls=":", lw=1.2, alpha=0.6)

    # Headline annotation
    min_pct  = fore_fc["yhat"].min()
    min_when = fore_fc.loc[fore_fc["yhat"].idxmin(), "ds"]
    crosses_in_window = bool((window_fc["yhat"] < THRESHOLD_PCT).any())
    first_below = (fore_fc.loc[fore_fc["yhat"] < THRESHOLD_PCT, "ds"].min()
                   if (fore_fc["yhat"] < THRESHOLD_PCT).any() else None)
    flag = "AT RISK" if crosses_in_window else "ON TARGET"
    flag_col = "#C62828" if crosses_in_window else "#2E7D32"
    note = (f"min forecast: {min_pct:.1f}% ({min_when.strftime('%b %Y')})"
            + (f"  ·  first crosses 80% in {first_below.strftime('%b %Y')}"
               if first_below is not None else ""))
    ax.text(0.02, 0.97, flag, transform=ax.transAxes, fontsize=10,
            fontweight="bold", color=flag_col, va="top",
            bbox=dict(facecolor="white", edgecolor=flag_col, lw=1.2,
                      boxstyle="round,pad=0.3"))
    ax.text(0.02, 0.88, note, transform=ax.transAxes, fontsize=8.5,
            color="#333", va="top",
            bbox=dict(facecolor="white", edgecolor="#CCC", lw=0.6,
                      boxstyle="round,pad=0.25"))

    ax.set_title(f"{antigen} — National coverage forecast (% of 2024 baseline)",
                 fontsize=10.5, fontweight="bold")
    ax.set_ylabel("% of 2024 baseline", fontsize=9.5)
    ax.legend(fontsize=7.5, ncol=2, loc="lower left")
    ax.grid(alpha=0.15, ls="--")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))

    # Collect for headline summary
    at_risk_summary.append({
        "antigen": antigen,
        "min_forecast_pct": round(min_pct, 1),
        "min_when": min_when.strftime("%b %Y"),
        "crosses_80_in_6_12m": crosses_in_window,
        "first_below_80": first_below.strftime("%b %Y") if first_below is not None else None,
    })

fig.suptitle("Domain 1 — Which antigens fall below the 80% target in 6–12 months?\n"
             "National Prophet forecasts as % of 2024 baseline  ·  Red dashed line = 80% target",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig_d1_national_forecast.png", dpi=150, bbox_inches="tight")
plt.show()

# Headline summary table — answers the question for the national level
summary_nat = pd.DataFrame(at_risk_summary)
summary_nat.to_csv(f"{OUTPUT_DIR}/d1_national_at_risk_summary.csv", index=False)
print("\nNational summary:")
print(summary_nat.to_string(index=False))
print(f"\nFigure + summary saved: fig_d1_national_forecast.png, d1_national_at_risk_summary.csv")



National summary:
 antigen  min_forecast_pct min_when  crosses_80_in_6_12m first_below_80
     BCG              93.5 Jan 2027                False           None
  Penta1              88.6 Oct 2026                False           None
  Penta3              87.5 Jul 2026                False           None
Measles1              92.5 Jul 2026                False           None

Figure + summary saved: fig_d1_national_forecast.png, d1_national_at_risk_summary.csv


### 5.1 LGA-level antigen forecasts

The national chart hides the operational truth: a national average above 80%
masks LGAs that are already collapsing, and a national average just below 80%
contains LGAs ranging from catastrophic to comfortably on-target. Domain 1's
answer for programme planners has to be at the level where micro-plans are
written — the LGA.

The same Prophet configuration is fit independently to each LGA × antigen
time-series. Each LGA's forecast is expressed as % of its own 2024 baseline so
the 80% threshold is meaningful locally. The output is a ranked CSV of
LGA × antigen combinations projected to cross below 80% inside the next
6–12 months — that is the at-risk worklist for the EPI team.

**Run-time note.** ~774 LGAs × 4 antigens ≈ 3,000 Prophet fits. On Colab this is
roughly 8–12 minutes. Set `LGA_SAMPLE` below to a list of priority LGAs to
shorten the run, or leave as `None` to fit all.


In [12]:
# Cell 18b — LGA-level Prophet forecasting × all four antigens
# Maintains the same Prophet config used for the national run. Each LGA gets its
# own baseline (2024 mean) and its own 80%-of-baseline threshold. The output is a
# ranked at-risk worklist for the next 6–12 months.

from prophet import Prophet
import warnings, logging
warnings.filterwarnings("ignore")
logging.getLogger("prophet").setLevel(logging.WARNING)
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)

# Optional: limit to a sample of LGAs to speed the run (None = all LGAs)
LGA_SAMPLE = None
# e.g. LGA_SAMPLE = sorted(d.loc[d["state"].isin(["Sokoto","Kebbi","Zamfara"]), "lga"].unique())

# Aggregate raw DHIS2 to state × lga × month
lga_agg = (d.groupby(["state", "zone", "lga", "ds"])[list(antigen_ts.values())]
             .sum().reset_index().sort_values(["state","lga","ds"]).reset_index(drop=True))
print(f"LGA-level rows : {len(lga_agg):,}")
print(f"Unique LGAs    : {lga_agg['lga'].nunique():,}")

lga_universe = (lga_agg["lga"].unique() if LGA_SAMPLE is None
                else [l for l in lga_agg["lga"].unique() if l in set(LGA_SAMPLE)])
print(f"LGAs to model  : {len(lga_universe):,}")

def _fit_prophet_silent(df, periods=FORECAST_MONTHS):
    """Prophet fit with the Domain-1 config, silenced."""
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                daily_seasonality=False, interval_width=0.95,
                changepoint_prior_scale=0.05, seasonality_prior_scale=10)
    m.add_seasonality(name="semi_annual", period=182.5, fourier_order=3)
    m.fit(df)
    future = m.make_future_dataframe(periods=periods, freq="MS")
    return m.predict(future)

rows, errors = [], 0
total = len(lga_universe) * len(antigen_ts)
done  = 0
print(f"Fitting {total:,} Prophet models …")
for lga in lga_universe:
    sub = lga_agg[lga_agg["lga"] == lga]
    if len(sub) < 18:        # need enough history for a stable fit
        done += len(antigen_ts); continue
    state = sub["state"].iloc[0]; zone = sub["zone"].iloc[0]
    for antigen, count_col in antigen_ts.items():
        ts = sub[["ds", count_col]].rename(columns={count_col: "y"}).dropna()
        if ts["y"].sum() == 0 or len(ts) < 18:
            done += 1; continue
        baseline = ts.loc[ts["ds"].dt.year == 2024, "y"].mean()
        if not np.isfinite(baseline) or baseline <= 0:
            done += 1; continue
        try:
            fc = _fit_prophet_silent(ts)
        except Exception:
            errors += 1; done += 1; continue
        cutoff  = ts["ds"].max()
        fore    = fc[fc["ds"] > cutoff].copy()
        fore["yhat_pct"] = fore["yhat"] / baseline * 100
        fore["months_ahead"] = ((fore["ds"].dt.year  - cutoff.year) * 12
                              + (fore["ds"].dt.month - cutoff.month))
        window = fore[(fore["months_ahead"] >= 6) & (fore["months_ahead"] <= 12)]
        if window.empty:
            done += 1; continue
        min_pct    = window["yhat_pct"].min()
        min_when   = window.loc[window["yhat_pct"].idxmin(), "ds"]
        below      = fore[fore["yhat_pct"] < 80.0]
        first_below = below["ds"].min() if not below.empty else pd.NaT
        rows.append({
            "state": state, "zone": zone, "lga": lga, "antigen": antigen,
            "baseline_2024_doses_per_month": round(baseline, 1),
            "min_forecast_pct_6_12m": round(min_pct, 1),
            "min_forecast_when": min_when.strftime("%Y-%m"),
            "crosses_80_in_6_12m": bool((window["yhat_pct"] < 80.0).any()),
            "first_month_below_80": first_below.strftime("%Y-%m") if pd.notna(first_below) else None,
        })
        done += 1
        if done % 200 == 0:
            print(f"  progress: {done:,}/{total:,}")

lga_fc_df = pd.DataFrame(rows)
print(f"\nFitted: {len(lga_fc_df):,} LGA × antigen forecasts  ·  errors: {errors}")

# Ranked at-risk worklist (only LGAs projected below 80% in the window)
at_risk = (lga_fc_df[lga_fc_df["crosses_80_in_6_12m"]]
            .sort_values(["antigen", "min_forecast_pct_6_12m"])
            .reset_index(drop=True))
print(f"At-risk LGA × antigen combinations: {len(at_risk):,}")

# Save all forecasts (audit trail) + at-risk worklist (planner's view)
lga_fc_df.to_csv(f"{OUTPUT_DIR}/d1_lga_forecast_all.csv", index=False)
at_risk.to_csv(f"{OUTPUT_DIR}/d1_lga_at_risk_below_80pct.csv", index=False)
print(f"\nSaved: d1_lga_forecast_all.csv  ·  d1_lga_at_risk_below_80pct.csv")

print("\nTop-15 worst-projected LGA × antigen (lowest forecast in 6–12 month window):")
print(at_risk.head(15)[["state","lga","antigen","min_forecast_pct_6_12m",
                         "min_forecast_when","first_month_below_80"]].to_string(index=False))

print("\nBreakdown — at-risk LGA counts by antigen and state:")
print(at_risk.groupby(["antigen","state"]).size().unstack(fill_value=0))


LGA-level rows : 43,656
Unique LGAs    : 774
LGAs to model  : 774
Fitting 3,096 Prophet models …
  progress: 200/3,096
  progress: 400/3,096
  progress: 600/3,096
  progress: 800/3,096
  progress: 1,000/3,096
  progress: 1,200/3,096
  progress: 1,400/3,096
  progress: 1,800/3,096
  progress: 2,000/3,096
  progress: 2,200/3,096
  progress: 2,400/3,096
  progress: 2,600/3,096
  progress: 2,800/3,096
  progress: 3,000/3,096

Fitted: 2,657 LGA × antigen forecasts  ·  errors: 0
At-risk LGA × antigen combinations: 1,645

Saved: d1_lga_forecast_all.csv  ·  d1_lga_at_risk_below_80pct.csv

Top-15 worst-projected LGA × antigen (lowest forecast in 6–12 month window):
      state                                     lga antigen  min_forecast_pct_6_12m min_forecast_when first_month_below_80
      Niger      ni Chanchaga Local Government Area     BCG                 -1975.1           2026-07              2026-01
    Katsina         kt Bakori Local Government Area     BCG                 -1770.1      

### 5.2 Antigen-specific coverage forecasts — reading the chart

**Research question answered.** *Which routine immunisation antigens are projected to fall below the 80% national coverage target in the next 6–12 months at LGA and ward levels?* The composite figure below answers this at three resolutions in a single read:

| Zone | What it shows | How to read it |
| --- | --- | --- |
| **Top row — severity distribution** | One bar chart per antigen. Bars count the number of at-risk LGAs in each severity band (Collapsed / Critical / Severe / Warning), based on the worst projected coverage inside the 6–12 month decision window. | Compare the **Severe (40–70%) and Warning (70–80%)** bars across antigens — these are the actionable cases. Antigens with taller Severe bars carry the highest concentration of meaningful risk. |
| **Middle row — state distribution** | Horizontal bars showing the top 8 states by count of LGAs with a credible at-risk signal. | States appearing across multiple antigens are the priority for cross-cutting intervention. Bar colour shows NPHCDA priority tier: Tier 1 red (Sokoto, Kebbi, Zamfara), Tier 2 orange (Kano, Kaduna, Niger, Katsina), slate for others. |
| **Bottom row — illustrative LGAs** | Ten named LGAs per antigen, stratified across the Critical / Severe / Warning bands (three each plus an extra in Severe). The bar shows the shortfall from the LGA's worst projected coverage to the 80% target. | The **dashed red line at 80%** is the national target. Bar length is the size of the projected gap. Worst LGAs sit at the top; bar colour matches the severity band. |

**Severity bands.**

- **Collapsed (<0%)** — Prophet forecast diverges into negative territory. This is a model pathology that affects LGAs with very low monthly dose counts (sparse history); reported transparently and excluded from the state and LGA rankings.
- **Critical (10–40%)** — credible forecast of severe coverage collapse.
- **Severe (40–70%)** — substantial projected shortfall against the 80% target.
- **Warning (70–80%)** — marginal projected shortfall; the LGA is on the threshold.

**Why this view replaces the earlier one.** The previous trajectory plot stacked eight visual channels per row (pink shading, amber band, sparkline, threshold line, lead-time dot, lowest-month square, right-edge percentage, tier-coloured name) and the 80% target — the *only* operationally important reference value — was lost in the fill. By moving severity to discrete bands and showing the gap to 80% as a bar, the threshold becomes the primary visual anchor on every row of the LGA panel. The state-level rollup in the middle row also makes the population-level answer explicit: where in Nigeria should NPHCDA concentrate its operational response?

**Ward-level extension** is on the v2 roadmap; see §5.3.


In [13]:
# Cell 18c — Domain 1 composite figure: severity / state / LGA roll-up
# Replaces the trajectory chart. Directly answers the research question by
# showing, for each antigen, (a) the severity distribution across at-risk LGAs,
# (b) the top 8 states by count of at-risk LGAs, and (c) ten illustrative LGAs
# stratified across the Critical / Severe / Warning bands.

# ─── Helper: clean LGA name ─────────────────────────────────────────────
def _clean_lga(name):
    """Return a short, presentable LGA label."""
    s = str(name).strip()
    for suffix in (" Local Government Area", " LGA"):
        if s.lower().endswith(suffix.lower()):
            s = s[: -len(suffix)]
    parts = s.split(None, 1)
    if len(parts) > 1 and len(parts[0]) == 2 and parts[0].islower():
        s = parts[1]
    return s.strip()

# ─── Configuration ──────────────────────────────────────────────────────
SEV_BINS   = [-float("inf"), 0, 40, 70, 80, float("inf")]
SEV_LABELS = ["Collapsed", "Critical", "Severe", "Warning", "Safe"]
SEV_COLORS = {
    "Collapsed": "#7A0F0F",
    "Critical":  "#A50026",
    "Severe":    "#D73027",
    "Warning":   "#F46D43",
    "Safe":      "#FDAE61",
}

# State priority tier
STATE_TIER = {
    "Sokoto":"Tier 1", "Kebbi":"Tier 1", "Zamfara":"Tier 1",
    "Kano":"Tier 2", "Kaduna":"Tier 2", "Niger":"Tier 2", "Katsina":"Tier 2",
}
TIER_FILL = {"Tier 1":"#C53030", "Tier 2":"#DD6B20", "Other":"#475569"}

ANTIGEN_COLOR = {
    "BCG":      "#1565C0",
    "Penta1":   "#2E7D32",
    "Penta3":   "#E65100",
    "Measles1": "#880E4F",
}

# Working frame: assign severity band + cleaned LGA name
work = at_risk.copy()
work["lga_clean"] = work["lga"].apply(_clean_lga)
work["severity"]  = pd.cut(work["min_forecast_pct_6_12m"],
                            bins=SEV_BINS, labels=SEV_LABELS)

# ─── Figure layout: 3 rows × 4 antigen columns ──────────────────────────
fig = plt.figure(figsize=(18, 14), facecolor="white")
gs  = fig.add_gridspec(
    nrows=3, ncols=4,
    height_ratios=[1.1, 1.6, 2.4],
    hspace=0.60, wspace=0.55,
    left=0.05, right=0.97, top=0.91, bottom=0.09,
)

for col, antigen in enumerate(["BCG", "Penta1", "Penta3", "Measles1"]):
    a_color = ANTIGEN_COLOR[antigen]
    sub     = work[work["antigen"] == antigen].copy()
    n_total = len(sub)

    # ── Zone A: severity distribution ─────────────────────────────────
    ax_a = fig.add_subplot(gs[0, col]); ax_a.set_facecolor("#FAFBFC")
    sev_counts = (sub["severity"].value_counts()
                   .reindex(SEV_LABELS, fill_value=0).drop("Safe"))
    x_pos = np.arange(len(sev_counts))
    ax_a.bar(x_pos, sev_counts.values,
              color=[SEV_COLORS[s] for s in sev_counts.index],
              edgecolor="white", lw=0.6, width=0.78, zorder=4)
    for i, v in enumerate(sev_counts.values):
        ax_a.text(i, v + max(sev_counts.values)*0.04, f"{int(v)}",
                   ha="center", va="bottom", fontsize=10,
                   fontweight="bold", color="#222")
    ax_a.set_xticks(x_pos)
    ax_a.set_xticklabels(sev_counts.index, fontsize=8.5)
    ax_a.set_ylabel("Number of LGAs", fontsize=9)
    ax_a.tick_params(axis="y", labelsize=8)
    ax_a.set_ylim(0, max(sev_counts.values)*1.18)
    ax_a.grid(axis="y", alpha=0.15, ls="--", zorder=0)
    for s in ax_a.spines.values():
        s.set_color("#AAA"); s.set_linewidth(0.5)
    ax_a.spines["top"].set_visible(False); ax_a.spines["right"].set_visible(False)
    ax_a.set_title(f"{antigen}    ·    {n_total} LGAs at-risk nationally",
                    fontsize=12.5, fontweight="bold",
                    color=a_color, pad=10, loc="center")
    credible_n = sev_counts.drop("Collapsed", errors="ignore").sum()
    n_le70 = sev_counts.get("Critical", 0) + sev_counts.get("Severe", 0)
    ax_a.text(0.5, -0.25,
              f"{int(credible_n)} LGAs with credible severity signal "
              f"({n_le70} \u2264 70%)",
              transform=ax_a.transAxes, ha="center", va="top",
              fontsize=8.5, color="#666", style="italic")

    # ── Zone B: top-8 states by credible count ────────────────────────
    ax_b = fig.add_subplot(gs[1, col]); ax_b.set_facecolor("#FAFBFC")
    sub_credible = sub[sub["min_forecast_pct_6_12m"] >= 0]
    state_counts = (sub_credible.groupby("state").size()
                     .sort_values(ascending=True).tail(8))
    if len(state_counts) > 0:
        bar_colors_b = [TIER_FILL.get(STATE_TIER.get(s, "Other"), "#475569")
                         for s in state_counts.index]
        y_pos = np.arange(len(state_counts))
        ax_b.barh(y_pos, state_counts.values, color=bar_colors_b,
                   edgecolor="white", lw=0.5, height=0.65, zorder=4)
        for i, v in enumerate(state_counts.values):
            ax_b.text(v + max(state_counts.values)*0.025, i, f"{v}",
                       va="center", ha="left", fontsize=9,
                       color="#222", fontweight="bold")
        ax_b.set_yticks(y_pos)
        ax_b.set_yticklabels(state_counts.index, fontsize=9.5)
        ax_b.set_xlim(0, max(state_counts.values)*1.18)
    ax_b.set_xlabel("LGAs with credible at-risk signal", fontsize=9)
    ax_b.tick_params(axis="x", labelsize=8)
    ax_b.grid(axis="x", alpha=0.15, ls="--", zorder=0)
    for s in ax_b.spines.values():
        s.set_color("#AAA"); s.set_linewidth(0.5)
    ax_b.spines["top"].set_visible(False); ax_b.spines["right"].set_visible(False)
    ax_b.set_title("Top 8 states by at-risk count",
                    fontsize=10, fontweight="bold",
                    color="#1F3864", loc="left", pad=4)

    # ── Zone C: top-10 LGAs stratified across severity bands ──────────
    ax_c = fig.add_subplot(gs[2, col]); ax_c.set_facecolor("#FAFBFC")
    top10 = sub_credible.sort_values("min_forecast_pct_6_12m", ascending=True)
    crit = top10[top10["min_forecast_pct_6_12m"].between(10, 40, inclusive="left")].head(3)
    sev_ = top10[top10["min_forecast_pct_6_12m"].between(40, 70, inclusive="left")].head(4)
    warn = top10[top10["min_forecast_pct_6_12m"].between(70, 80, inclusive="left")].head(3)
    top10 = pd.concat([crit, sev_, warn]).iloc[::-1]   # worst at top
    n = len(top10)

    if n > 0:
        ax_c.axvline(80, color="#A50026", lw=1.4, ls="--",
                      alpha=0.9, zorder=2,
                      label="80% national target")
        for i, (_, r) in enumerate(top10.iterrows()):
            p     = float(r["min_forecast_pct_6_12m"])
            sev   = str(r["severity"])
            color = SEV_COLORS.get(sev, "#888")
            # Shortfall bar from p to 80
            ax_c.barh(i, 80 - p, left=p, height=0.55,
                       color=color, alpha=0.85, edgecolor="white",
                       lw=0.6, zorder=4)
            # Dot at the worst value
            ax_c.scatter([p], [i], s=110, color=color,
                          edgecolor="white", lw=1.2, zorder=10)
            # Label: inside bar if wide enough, else outside
            bar_width = 80 - p
            if bar_width >= 20:
                ax_c.text((p + 80)/2, i, f"{p:.0f}%",
                           va="center", ha="center", fontsize=8.5,
                           fontweight="bold", color="white")
            else:
                ax_c.text(82, i, f"{p:.0f}%",
                           va="center", ha="left", fontsize=8.5,
                           fontweight="bold", color=color)

        y_labels = [f"{r['lga_clean']} \u00B7 {r['state']}"
                     for _, r in top10.iterrows()]
        ax_c.set_yticks(range(n))
        ax_c.set_yticklabels(y_labels, fontsize=8.5)
        ax_c.set_ylim(-0.6, n - 0.4)
        # Tier badge to the left of each label
        for i, (_, r) in enumerate(top10.iterrows()):
            tier = STATE_TIER.get(r["state"], "Other")
            ax_c.scatter(-0.02, i, s=80,
                          color=TIER_FILL[tier], marker="s",
                          transform=ax_c.get_yaxis_transform(),
                          clip_on=False, zorder=15)

    ax_c.set_xlim(0, 105)
    ax_c.set_xticks([0, 20, 40, 60, 80, 100])
    ax_c.set_xticklabels(["0%", "20%", "40%", "60%", "80%", "100%"], fontsize=8.5)
    ax_c.set_xlabel("Worst projected coverage in 6\u201312 month window\n"
                     "(% of 2024 baseline)", fontsize=9)
    ax_c.tick_params(axis="y", length=0, pad=14)
    ax_c.grid(axis="x", alpha=0.15, ls="--", zorder=0)
    for s in ax_c.spines.values():
        s.set_color("#AAA"); s.set_linewidth(0.5)
    ax_c.spines["top"].set_visible(False); ax_c.spines["right"].set_visible(False)
    ax_c.set_title("Top 10 illustrative LGAs", fontsize=10,
                    fontweight="bold", color="#1F3864", loc="left", pad=4)

# ─── Suptitle and legends ───────────────────────────────────────────────
fig.suptitle(
    "Domain 1 \u00B7 LGA-level coverage forecasts at risk of crossing the 80% national target\n"
    "Operational decision window: months 6\u201312 ahead. "
    "Top row: severity distribution per antigen. "
    "Middle row: states with most at-risk LGAs. "
    "Bottom row: top illustrative LGAs.",
    fontsize=13, fontweight="bold", color="#1F3864", y=0.98
)

sev_h = [
    mpatches.Patch(color=SEV_COLORS["Collapsed"],
                    label="Collapsed (<0%) \u2014 model pathology"),
    mpatches.Patch(color=SEV_COLORS["Critical"], label="Critical (0\u201340%)"),
    mpatches.Patch(color=SEV_COLORS["Severe"],   label="Severe (40\u201370%)"),
    mpatches.Patch(color=SEV_COLORS["Warning"],  label="Warning (70\u201380%)"),
]
tier_h = [
    mpatches.Patch(color=TIER_FILL["Tier 1"], label="Tier 1 state"),
    mpatches.Patch(color=TIER_FILL["Tier 2"], label="Tier 2 state"),
    mpatches.Patch(color=TIER_FILL["Other"],  label="Other state"),
]
target_h = [
    plt.Line2D([0], [0], color="#A50026", lw=1.6, ls="--",
                label="80% national coverage target"),
]
fig.legend(handles=sev_h + target_h, loc="lower left",
            ncol=5, fontsize=8.5, frameon=True, framealpha=0.95,
            edgecolor="#CCC", bbox_to_anchor=(0.05, 0.0),
            title="Severity bands (based on worst projected coverage)",
            title_fontsize=9)
fig.legend(handles=tier_h, loc="lower right",
            ncol=3, fontsize=8.5, frameon=True, framealpha=0.95,
            edgecolor="#CCC", bbox_to_anchor=(0.97, 0.0),
            title="State priority tier", title_fontsize=9)

plt.savefig(f"{OUTPUT_DIR}/fig_d1_lga_at_risk_composite.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: fig_d1_lga_at_risk_composite.png")


Saved: fig_d1_lga_at_risk_composite.png


### 5.3 Ward-level extension (note)

The same Prophet configuration applies at the ward level, but the current DHIS2
extract aggregates to LGA. To produce ward-level forecasts, pull a ward-tagged
DHIS2 extract with the same monthly granularity (`ds`, `state`, `lga`, `ward`,
plus the count columns used here) and replace the `lga_agg` aggregation key
with `["state", "zone", "lga", "ward", "ds"]`. The fit loop, baseline computation
and at-risk worklist code generalise without change — the only practical
constraint is that wards typically have lower monthly counts and shorter useful
history, which inflates prediction intervals. For wards with fewer than 18
months of non-zero history, the pipeline falls through to no-forecast and they
should be analysed by direct trend inspection instead.


---
## Section 6 — Domain 2: Dropout & Completion Dynamics

### 6.1 Prophet forecast of dropout rates

In [14]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7), facecolor=BG)
all_dropout_fc = []
for ax, (col, label, color) in zip(axes, [
    ("dropout_p1p3", "Penta1 → Penta3 Dropout Rate (%)", "#E65100"),
    ("dropout_p1m1", "Penta1 → Measles1 Dropout Rate (%)", "#880E4F"),
    ("dropout_m1m2", "Measles1 → Measles2 Dropout Rate (%)", "#4A148C"),
]):
    ts_d = nat[["ds", col]].rename(columns={"ds":"ds", col:"y"}).dropna()
    fc_d, _ = run_prophet(ts_d, col)
    all_dropout_fc.append((col, fc_d))
    cutoff = ts_d["ds"].max()
    fore   = fc_d[fc_d["ds"] > cutoff]; hist = fc_d[fc_d["ds"] <= cutoff]
    ax.set_facecolor(BG)
    ax.scatter(ts_d["ds"], ts_d["y"], s=16, color=color, alpha=0.60, zorder=4, label="Observed")
    ax.plot(hist["ds"], hist["yhat"], lw=2.2, color=color, alpha=0.9, label="Fitted")
    ax.plot(fore["ds"], fore["yhat"], "--", lw=2.4, color=color)
    ax.fill_between(fore["ds"], fore["yhat_lower"], fore["yhat_upper"],
                    alpha=0.15, color=color, label="95% PI")
    lo80 = fore["yhat"]-(fore["yhat"]-fore["yhat_lower"])*0.53
    hi80 = fore["yhat"]+(fore["yhat_upper"]-fore["yhat"])*0.53
    ax.fill_between(fore["ds"], lo80, hi80, alpha=0.28, color=color, label="80% PI")
    ax.axhline(0, color="#333", ls=":", lw=1.4, alpha=0.7)
    ax.axvline(cutoff, color="#888", ls=":", lw=1.1)
    ax.set_title(f"Domain 2 — {label}\nProphet Forecast", fontsize=10.5, fontweight="bold")
    ax.set_ylabel(label, fontsize=9); ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.15, ls="--")
fig.suptitle("Domain 2 — Dropout Rate Forecasting\n80% & 95% Prediction Intervals | Nigeria",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig_d2_dropout_forecast.png", dpi=200, bbox_inches="tight")
plt.show()


### 6.2 LASSO cross-validated feature importance

In [15]:
mod = pd.read_csv(MODEL_PATH)
lasso_features = [
    "anc_4plus","delivered_in_hf","pct_c12to23_vax_card_seen",
    "pct_cu5_birth_registered","pct_using_improved_water_source",
    "pct_cu5_slept_under_itn","pct_problem_accessing_hfdistance",
    "pct_cu5_stunted","pct_women_no_education","pct_lowest_wealth_quintile",
    "total_fertility_rate","pct_urban","pct_muslim",
    "pct_women_with_mobile_phone","pct_media_at_least_once_week",
    "pct_women_curr_employed","pct_women_say_wife_beating_justified",
]

state_dropout_2024 = (agg[agg["year"]==2024]
                      .groupby("state")[["dropout_p1p3","dropout_p1m1","dropout_m1m2"]]
                      .mean().reset_index())
state_dropout_2024["state"] = state_dropout_2024["state"].str.strip().str.title()
mod["state_key"] = mod["state_name"].str.strip().str.title()
merged = state_dropout_2024.merge(mod[["state_key"]+lasso_features],
                                   left_on="state", right_on="state_key", how="inner")

X   = merged[lasso_features].fillna(merged[lasso_features].median())
X_sc = StandardScaler().fit_transform(X)
results_lasso = {}
for target in ["dropout_p1p3","dropout_p1m1","dropout_m1m2"]:
    y = merged[target].fillna(merged[target].median()).values
    lasso = LassoCV(cv=5, random_state=42, max_iter=5000)
    lasso.fit(X_sc, y)
    coefs = pd.Series(lasso.coef_, index=lasso_features).abs().sort_values(ascending=False)
    results_lasso[target] = coefs
    print(f"\n{target} — top predictors:")
    print(coefs[coefs>0].head(8).to_string())



dropout_p1p3 — top predictors:
pct_cu5_stunted                 2.205494
pct_media_at_least_once_week    0.947152
pct_lowest_wealth_quintile      0.864162
pct_c12to23_vax_card_seen       0.433729
pct_women_with_mobile_phone     0.404058
anc_4plus                       0.319753
pct_cu5_slept_under_itn         0.311284
pct_women_no_education          0.126488

dropout_p1m1 — top predictors:
pct_women_curr_employed             4.537736
delivered_in_hf                     4.055796
pct_muslim                          2.704359
pct_cu5_slept_under_itn             1.797050
pct_problem_accessing_hfdistance    1.395267

dropout_m1m2 — top predictors:
pct_muslim                          8.067716
pct_problem_accessing_hfdistance    5.204603
pct_cu5_slept_under_itn             2.842275
pct_women_curr_employed             1.484467
pct_cu5_stunted                     0.308062


In [16]:
fig, axes = plt.subplots(1, 3, figsize=(22, 8), facecolor=BG)
target_labels = {
    "dropout_p1p3":   "Penta1 → Penta3",
    "dropout_p1m1":   "Penta1 → Measles1",
    "dropout_m1m2":   "Measles1 → Measles2",
}
bar_colors = {
    "dropout_p1p3":   "#E65100",
    "dropout_p1m1":   "#880E4F",
    "dropout_m1m2":   "#4A148C",
}
for ax, (target, coefs) in zip(axes, results_lasso.items()):
    top = coefs[coefs>0].head(12)
    ax.barh(range(len(top)), top.values, color=bar_colors[target], height=0.65, alpha=0.85)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels([f.replace("pct_","").replace("_"," ").title() for f in top.index], fontsize=9.5)
    ax.set_xlabel("|LASSO Coefficient|", fontsize=10)
    ax.set_title(f"{target_labels[target]} Dropout — LASSO Feature Importance",
                  fontsize=10.5, fontweight="bold")
    ax.grid(axis="x", alpha=0.2, ls="--"); ax.set_facecolor(BG)
    for i, v in enumerate(top.values):
        ax.text(v+0.001, i, f"{v:.3f}", va="center", fontsize=8.5)
    ax.invert_yaxis()
fig.suptitle("Domain 2 — Drivers of Vaccination Dropout\nLASSO | Nigeria 2024",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig_d2_lasso_importance.png", dpi=200, bbox_inches="tight")
plt.show()


### 6.3 State dropout heatmaps — Penta1→Penta3 and Measles1→Measles2 (observed + forecast)


In [17]:
# Cell 30 — State × Year dropout heatmaps (imputed observed + Prophet forecast)
# Two panels:
#   Left  — Penta1 → Penta3 dropout
#   Right — Measles1 → Measles2 dropout
# Each panel concatenates imputed annual means (2021–2025) with calendar-year
# means of monthly Prophet forecasts (2026–2027).
#
# Visual conventions:
#   - Bold vertical divider between observed and forecast columns
#   - Italic-navy column ticks for forecast years
#   - Diagonal hatching on cells whose underlying state-month rows were
#     imputed (ensemble KNN + IterativeImputer + PMM snap)
#   - Footer caveat documents both the forecast and the imputation method

import warnings
warnings.filterwarnings("ignore")

DROPOUT_PANELS = [
    ("dropout_p1p3", "Penta1 → Penta3",     "Penta1→Penta3 Dropout Rate (%)"),
    ("dropout_m1m2", "Measles1 → Measles2", "Measles1→Measles2 Dropout Rate (%)"),
]

# Use imputed frame for the historical block
src_df = agg_imputed if "agg_imputed" in globals() else agg

last_obs_date = src_df["ds"].max()
forecast_end  = pd.Timestamp("2027-12-01")
months_to_end = ((forecast_end.year  - last_obs_date.year) * 12
               + (forecast_end.month - last_obs_date.month))

def build_state_year_tables(metric):
    """Return (pivot_values, pivot_imputed_flag, hist_years, fc_years)."""
    hist = src_df[["state", "year", metric, "is_imputed"]].copy()
    hist_grp = (hist.groupby(["state", "year"])
                     .agg(value=(metric, "mean"),
                          any_imputed=("is_imputed", "any"))
                     .reset_index())
    hist_years = sorted(hist_grp["year"].unique())

    fc_rows = []
    for state in sorted(src_df["state"].unique()):
        ts = (src_df[src_df["state"] == state][["ds", metric]]
                .rename(columns={metric: "y"}).dropna())
        if len(ts) < 18:
            continue
        try:
            fc_state, _ = run_prophet(ts, f"{metric}_{state}",
                                       periods=max(months_to_end, FORECAST_MONTHS))
        except Exception:
            continue
        fut = fc_state[fc_state["ds"] > last_obs_date].copy()
        fut["year"] = fut["ds"].dt.year
        yr = (fut.groupby("year")["yhat"].mean().reset_index()
                  .rename(columns={"yhat": "value"}))
        yr["state"]       = state
        yr["any_imputed"] = False
        fc_rows.append(yr[["state", "year", "value", "any_imputed"]])

    fc_df = (pd.concat(fc_rows, ignore_index=True) if fc_rows
             else pd.DataFrame(columns=["state","year","value","any_imputed"]))
    fc_years = [y for y in sorted(fc_df["year"].unique())
                if y > last_obs_date.year]
    fc_df = fc_df[fc_df["year"].isin(fc_years)]

    combined = (pd.concat([hist_grp, fc_df], ignore_index=True)
                   .drop_duplicates(["state", "year"], keep="first"))

    piv_val = combined.pivot(index="state", columns="year", values="value").round(1)
    piv_imp = combined.pivot(index="state", columns="year", values="any_imputed").fillna(False)
    piv_val.columns = [int(c) for c in piv_val.columns]
    piv_imp.columns = [int(c) for c in piv_imp.columns]
    piv_val = piv_val.reindex(sorted(piv_val.columns), axis=1)
    piv_imp = piv_imp.reindex(piv_val.columns, axis=1)
    return piv_val, piv_imp, hist_years, fc_years

def draw_dropout_heatmap(ax, piv_val, piv_imp, hist_years, fc_years, *,
                          cbar_label, title, vmin=-30, vmax=30):
    n_hist = sum(1 for c in piv_val.columns if c in hist_years)
    n_fc   = sum(1 for c in piv_val.columns if c in fc_years)

    # Fixed vmin/vmax (Option A): same colour means the same dropout magnitude
    # in both panels, so reviewers can compare Penta1→Penta3 and
    # Measles1→Measles2 directly. Values outside [vmin, vmax] are clipped
    # to the bar extremes — that is the desired behaviour because anything
    # beyond ±30% dropout is an outlier worth visual attention regardless of
    # which side. We add the "extend" arrows so reviewers see at a glance
    # that the scale truncates.
    sns.heatmap(piv_val, ax=ax, cmap="RdYlGn_r", center=0,
                 vmin=vmin, vmax=vmax,
                 linewidths=0.4, linecolor="#ddd",
                 cbar_kws={"label": cbar_label, "shrink": 0.85,
                            "extend": "both"},
                 annot=True, fmt=".1f", annot_kws={"size": 7})

    # Hatch overlay on imputed cells
    n_imp = 0
    for j, col in enumerate(piv_val.columns):
        for i, state in enumerate(piv_val.index):
            try:
                if bool(piv_imp.iloc[i, j]):
                    ax.add_patch(plt.Rectangle((j, i), 1, 1, fill=False,
                                                 hatch="////", edgecolor="#1F2937",
                                                 lw=0, alpha=0.55, zorder=4))
                    n_imp += 1
            except (IndexError, KeyError):
                continue

    if n_fc > 0:
        ax.axvline(n_hist, color="#0F172A", lw=2.2, alpha=0.9, zorder=5)

    ax.text(n_hist / 2, -0.6,
            f"OBSERVED  ·  {hist_years[0]}–{hist_years[-1]}",
            ha="center", va="bottom", fontsize=9.5, fontweight="bold",
            color="#0F172A", transform=ax.transData)
    if n_fc > 0:
        ax.text(n_hist + n_fc / 2, -0.6,
                f"FORECAST  ·  {fc_years[0]}–{fc_years[-1]}",
                ha="center", va="bottom", fontsize=9.5, fontweight="bold",
                color="#1E3A8A", transform=ax.transData)

    for tick, col in zip(ax.get_xticklabels(), piv_val.columns):
        if col in fc_years:
            tick.set_color("#1E3A8A")
            tick.set_fontweight("bold")
            tick.set_fontstyle("italic")

    ax.set_title(title, fontsize=11.5, fontweight="bold",
                  color="#0F172A", pad=28, loc="left")
    ax.set_xlabel(""); ax.set_ylabel("State", fontsize=10)
    return n_imp

# Build both pivots
pv_p, pi_p, hy_p, fy_p = build_state_year_tables("dropout_p1p3")
pv_m, pi_m, hy_m, fy_m = build_state_year_tables("dropout_m1m2")

fig, axes = plt.subplots(1, 2, figsize=(22, 12), facecolor=BG)
n_imp_p = draw_dropout_heatmap(axes[0], pv_p, pi_p, hy_p, fy_p,
                                cbar_label="Penta1->Penta3 Dropout (%)",
                                title="Penta1 -> Penta3 Dropout by State and Year")
n_imp_m = draw_dropout_heatmap(axes[1], pv_m, pi_m, hy_m, fy_m,
                                cbar_label="Measles1->Measles2 Dropout (%)",
                                title="Measles1 -> Measles2 Dropout by State and Year")

fig.suptitle("Domain 2 — Dropout Heatmaps by State and Year  ·  "
              "Imputed Observed + Prophet Forecast",
              fontsize=13.5, fontweight="bold", color="#0F172A", y=0.99)

# Legend strip for the hatch pattern
from matplotlib.patches import Patch
hatch_patch = Patch(facecolor="white", edgecolor="#1F2937", hatch="////",
                     label="Hatched cells = state-month values imputed "
                            "(ensemble KNN + BayesianRidge with PMM snap)")
fig.legend(handles=[hatch_patch], loc="lower center", ncol=1,
            fontsize=9, frameon=True, framealpha=0.95, edgecolor="#CCC",
            bbox_to_anchor=(0.5, 0.025))

fig.text(0.5, 0.005,
          "Colour scale is fixed at ±30% in both panels so equivalent colours "
          "represent equivalent dropout magnitudes; values beyond ±30% clip "
          "to the bar extremes. Forecast columns are calendar-year means of "
          "monthly Prophet point forecasts per state · negative values denote "
          "upward visits between the two doses.",
          ha="center", fontsize=9, color="#475569", style="italic")

plt.tight_layout(rect=[0, 0.05, 1, 0.97])
plt.savefig(f"{OUTPUT_DIR}/fig_d2_dropout_heatmap.png", dpi=150,
             bbox_inches="tight", facecolor=BG)
plt.show()
print(f"Saved dual-panel heatmap  ·  P1->P3: {n_imp_p} imputed cells  "
       f"·  M1->M2: {n_imp_m} imputed cells")


Saved dual-panel heatmap  ·  P1->P3: 10 imputed cells  ·  M1->M2: 10 imputed cells


---
## Section 7 — Domain 5-II: Community Archetype Classification

**Hierarchical Agglomerative Clustering (HAC)** on 15 contextual features. Ward linkage minimises within-cluster variance. Optimal k=5 clusters represent distinct community profiles explaining persistent zero-dose.

In [ ]:
cluster_features = [
    "zero_dose_2024","pct_urban","pct_problem_accessing_hfdistance",
    "pct_women_no_education","pct_lowest_wealth_quintile","pct_muslim",
    "pct_severely_food_insecure","total_fertility_rate","pct_cu5_stunted",
    "pct_women_with_mobile_phone","anc_4plus","delivered_in_hf",
    "pct_cu5_birth_registered","pct_women_say_wife_beating_justified",
    "pct_women_moved_current_res_past5yrs",
]

X_cl  = mod[cluster_features].fillna(mod[cluster_features].median()).values
X_sc2 = StandardScaler().fit_transform(X_cl)
state_labels = mod["state_name"].values
print(f"Clustering {len(X_cl)} states on {len(cluster_features)} features.")


### 7.1 Optimal cluster selection (silhouette + Calinski-Harabász)

In [ ]:
sil_scores, ch_scores = [], []
k_range = range(2, 10)
for k in k_range:
    hac    = AgglomerativeClustering(n_clusters=k, linkage="ward")
    labels = hac.fit_predict(X_sc2)
    sil_scores.append(silhouette_score(X_sc2, labels))
    ch_scores.append(calinski_harabasz_score(X_sc2, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=BG)
axes[0].plot(list(k_range), sil_scores, "o-", color="#1565C0", lw=2.2, ms=7)
axes[0].axvline(N_CLUSTERS, color="#C62828", ls="--", lw=1.5, label=f"Selected k={N_CLUSTERS}")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Silhouette Score")
axes[0].set_title("Silhouette Score vs k", fontweight="bold")
axes[0].legend(); axes[0].grid(alpha=0.2, ls="--"); axes[0].set_facecolor(BG)

axes[1].plot(list(k_range), ch_scores, "s-", color="#2E7D32", lw=2.2, ms=7)
axes[1].axvline(N_CLUSTERS, color="#C62828", ls="--", lw=1.5, label=f"Selected k={N_CLUSTERS}")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Calinski-Harabász Score")
axes[1].set_title("Calinski-Harabász Score vs k", fontweight="bold")
axes[1].legend(); axes[1].grid(alpha=0.2, ls="--"); axes[1].set_facecolor(BG)

fig.suptitle("Domain 5-II — Optimal Cluster Selection", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig_d5_cluster_selection.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Silhouette at k=2: {sil_scores[0]:.3f}  k=5: {sil_scores[3]:.3f}")


### 7.2 Dendrogram

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

Z = linkage(X_sc2, method="ward")
fig, ax = plt.subplots(figsize=(18, 9), facecolor=BG)
dend = dendrogram(Z, labels=state_labels, ax=ax,
                  leaf_rotation=90, leaf_font_size=9,
                  color_threshold=Z[-N_CLUSTERS+1,2],
                  above_threshold_color="#AAAAAA")
ax.axhline(Z[-N_CLUSTERS+1,2], color="#333", ls="--", lw=1.5,
           label=f"Cut: k={N_CLUSTERS} clusters")
ax.set_title("Domain 5-II — Hierarchical Agglomerative Clustering Dendrogram\n"
             "Ward Linkage | Euclidean Distance | 15 Features | Nigeria 37 States",
             fontsize=12, fontweight="bold")
ax.set_ylabel("Ward Linkage Distance", fontsize=10)
ax.legend(fontsize=9); ax.grid(axis="y", alpha=0.15, ls="--"); ax.set_facecolor(BG)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig_d5_dendrogram.png", dpi=150, bbox_inches="tight")
plt.show()


### 7.3 Assign cluster labels and archetype names

In [ ]:
hac_final   = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage="ward")
cluster_ids = hac_final.fit_predict(X_sc2)
mod2        = mod.copy()
mod2["cluster_id"] = cluster_ids

# Sort clusters by zero-dose rate descending so Cluster A always carries the
# heaviest burden.
profile = mod2.groupby("cluster_id")[cluster_features].mean()
cluster_order = profile["zero_dose_2024"].sort_values(ascending=False).index.tolist()

# Build descriptive labels from each cluster's z-scored profile. The procedure:
#   1. Take the cluster's mean on every feature, z-score against the national mean.
#   2. Find which feature deviates most strongly from the national mean (positive
#      or negative).
#   3. Use that feature plus zero-dose rank to form a label.
# This guarantees the labels follow the actual data, not an a-priori narrative.

label_map = {
    # feature, sign-of-deviation -> label fragment
    ("pct_problem_accessing_hfdistance", +1): "Distance Barrier",
    ("pct_problem_accessing_hfdistance", -1): "Strong Access",
    ("pct_lowest_wealth_quintile",       +1): "High Poverty",
    ("pct_women_no_education",           +1): "Low Education",
    ("pct_women_no_education",           -1): "High Education",
    ("pct_urban",                        +1): "Urban Majority",
    ("pct_urban",                        -1): "Rural Majority",
    ("total_fertility_rate",             +1): "High Fertility",
    ("pct_women_with_mobile_phone",      +1): "High Digital Access",
    ("pct_women_with_mobile_phone",      -1): "Low Digital Access",
    ("anc_4plus",                        +1): "Strong ANC Coverage",
    ("anc_4plus",                        -1): "Weak ANC Coverage",
    ("delivered_in_hf",                  +1): "High Facility Delivery",
    ("delivered_in_hf",                  -1): "Low Facility Delivery",
    ("pct_cu5_stunted",                  +1): "High Stunting",
    ("pct_muslim",                       +1): "Predominantly Muslim",
}

z_profile = (profile - profile.mean()) / profile.std()
zd_rank_label = {0: "Highest Burden", 1: "High Burden",
                  2: "Moderate Burden", 3: "Lower Burden",
                  4: "Lowest Burden"}

archetype_assignment = {}
for rank, cid in enumerate(cluster_order):
    # Drop zero-dose from descriptor selection (we already use rank for it)
    descriptors = z_profile.loc[cid].drop(["zero_dose_2024"]).copy()
    # Find the two most-deviant features (by absolute z-score) that have a
    # corresponding label entry; build the secondary descriptor from them.
    chosen = []
    for f in descriptors.abs().sort_values(ascending=False).index:
        sign = +1 if descriptors[f] > 0 else -1
        if (f, sign) in label_map:
            chosen.append(label_map[(f, sign)])
        if len(chosen) >= 2:
            break
    secondary = ", ".join(chosen) if chosen else "Mixed profile"
    cluster_letter = chr(ord("A") + rank)
    archetype_assignment[cid] = (
        f"Cluster {cluster_letter} — {zd_rank_label[rank]}, {secondary}"
    )

mod2["archetype"] = mod2["cluster_id"].map(archetype_assignment)
archetypes_ordered = [archetype_assignment[k] for k in cluster_order]

# Palette: red (highest burden) -> green (lowest burden), aligned with the
# zero-dose tier scheme used in Domains 3 and 5.
_palette = ["#D73027", "#F46D43", "#FDAE61", "#1565C0", "#1A9850"]
ARCHETYPE_COLORS_NAMED = {a: c for a, c in zip(archetypes_ordered, _palette)}

print("Archetype labels (derived from cluster fingerprints):")
for a in archetypes_ordered:
    n = (mod2["archetype"]==a).sum()
    print(f"  {a}   (n={n})")

print("\nState archetype assignments:")
print(mod2[["state_name","zone_name","archetype","zero_dose_2024"]]
      .sort_values(["archetype","zero_dose_2024"],ascending=[True,False]).to_string(index=False))


### 7.4 Cluster fingerprint — premium archetype profile heatmap

An **archetype** is a reusable summary of a cluster — a typical profile of demographics, equity exposure, and service coverage that the states in the cluster share. Instead of designing 37 state-specific interventions, NPHCDA can design one intervention package per archetype and apply it consistently across the states that share that profile.

**How to read the fingerprint heatmap below.**

- Each **column** is one archetype. Columns are ordered left-to-right from highest zero-dose burden (Cluster A) to lowest (Cluster E).
- Each **row** is one variable from the clustering feature set.
- **Cell colour** shows how the cluster's mean compares to the national mean, in standard-deviation units. Red = above the national mean, blue = below. Saturated colours indicate stronger deviation.
- **Cell text** shows the raw mean for that cluster on that variable (percent, or fertility rate). This grounds the colour interpretation in interpretable units.
- The **coloured stripe on the left edge of each row** groups variables by category: outcome, demographic, equity barrier, service-readiness, and service-coverage.

A cluster is well-defined when the column shows a clear vertical pattern of red and blue cells — that is its "fingerprint". A cell that is white indicates the cluster is close to the national average on that variable, which means the variable is not a defining feature of that cluster.

**Cluster labels are descriptive, not interpretive.** Each label names what the data actually shows (e.g. *"High Poverty, High Fertility"*) rather than embedding an external assumption about conflict, nomadism, or other unmeasured attributes. Programmatic narratives that wish to overlay context (e.g. "Cluster C overlaps with the Lake Chad conflict zone") can do so explicitly in the briefing layer, with the data limitation acknowledged.


In [ ]:
# Variable ordering: outcome → demographic → equity barrier → service-readiness → service-coverage
key_vars = [
    ("zero_dose_2024",                  "Zero-Dose Rate (2024)",                 "Outcome"),
    ("pct_urban",                       "Urbanicity (% urban)",                  "Demographic"),
    ("pct_women_no_education",          "Women, no formal education",            "Equity barrier"),
    ("pct_lowest_wealth_quintile",      "Lowest wealth quintile",                "Equity barrier"),
    ("pct_problem_accessing_hfdistance","HTR (distance to HF a problem)",        "Equity barrier"),
    ("pct_muslim",                      "Muslim population",                     "Demographic"),
    ("total_fertility_rate",            "Total fertility rate",                  "Demographic"),
    ("pct_women_with_mobile_phone",     "Women with mobile phone",               "Service-readiness"),
    ("anc_4plus",                       "ANC 4+ visits",                         "Service-coverage"),
    ("delivered_in_hf",                 "Facility delivery",                     "Service-coverage"),
]

# Build z-score and raw-mean matrices (rows = variable, columns = archetype)
zmat = np.zeros((len(key_vars), len(archetypes_ordered)))
rmat = np.zeros_like(zmat)
n_per = []
for c, arch in enumerate(archetypes_ordered):
    n_per.append(int((mod2["archetype"]==arch).sum()))
    for r, (v, _, _) in enumerate(key_vars):
        nat_mean = mod2[v].mean(); nat_std = mod2[v].std()
        sub = mod2.loc[mod2["archetype"]==arch, v]
        zmat[r, c] = (sub.mean() - nat_mean) / nat_std if nat_std > 0 else 0
        rmat[r, c] = sub.mean()

# ─── FIGURE 1: archetype fingerprint heatmap ───────────────────────
fig = plt.figure(figsize=(15, 11), facecolor="white")
gs  = fig.add_gridspec(1, 2, width_ratios=[5, 0.18], wspace=0.02)
ax  = fig.add_subplot(gs[0, 0])
cax = fig.add_subplot(gs[0, 1])

import matplotlib.colors as mcolors
vmax = max(float(np.nanpercentile(np.abs(zmat), 98)), 1.5)
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
im   = ax.imshow(zmat, cmap="RdBu_r", norm=norm, aspect="auto")

# Cell annotation: raw mean
for r in range(len(key_vars)):
    for c in range(len(archetypes_ordered)):
        val = rmat[r, c]; z = zmat[r, c]
        txt_color = "white" if abs(z) > vmax*0.6 else "#111"
        s = f"{val:.1f}" if key_vars[r][0] == "total_fertility_rate" else f"{val:.0f}%"
        ax.text(c, r, s, ha="center", va="center",
                fontsize=10, fontweight="bold", color=txt_color)

# Row labels
ax.set_yticks(np.arange(len(key_vars)))
ax.set_yticklabels([k[1] for k in key_vars], fontsize=10.5)

# Column labels — Option B: descriptive labels above the heatmap, short
# letter ticks below. Reader sees full archetype context on first glance
# (top), and keeps a compact column anchor while scanning deep rows (bottom).
import textwrap
def _wrap_label(arch, n):
    parts = arch.split(" — ", 1)
    head = parts[0]                         # "Cluster A"
    rest = parts[1] if len(parts) > 1 else ""
    rest_wrapped = "\n".join(textwrap.wrap(rest, width=22))
    return f"{head}\n{rest_wrapped}\n(n = {n})"

def _short(arch):
    """Return the short cluster name only — 'Cluster A', 'Cluster B', etc.
    Drops the descriptor that's already shown on the top axis."""
    return arch.split(" — ", 1)[0]          # "Cluster A"

n_clusters = len(archetypes_ordered)

# Bottom — full cluster names ('Cluster A', 'Cluster B', …) for naming
# consistency with the top axis. The bottom row anchors column identity
# when the reader is scanning deep rows of the heatmap.
ax.set_xticks(np.arange(n_clusters))
ax.set_xticklabels([_short(a) for a in archetypes_ordered],
                    fontsize=11, fontweight="bold", color="#0F172A")
ax.tick_params(axis="x", length=0, pad=6)
ax.tick_params(axis="y", length=0)

# Top — full descriptive labels on a twin x-axis sharing the same scale
ax_top = ax.secondary_xaxis("top")
ax_top.set_xticks(np.arange(n_clusters))
ax_top.set_xticklabels([_wrap_label(a, n_per[i])
                          for i, a in enumerate(archetypes_ordered)],
                        fontsize=8.5, color="#0F172A")
ax_top.tick_params(axis="x", length=0, pad=8)

# Left-edge category-stripe colours
cat_colors = {
    "Outcome":           "#A32D2D",
    "Demographic":       "#5A6F8C",
    "Equity barrier":    "#BA7517",
    "Service-readiness": "#1A6E8C",
    "Service-coverage":  "#1D9E75",
}
for i, (v, _, cat) in enumerate(key_vars):
    ax.add_patch(plt.Rectangle((-0.55, i-0.5), 0.06, 1.0,
                                color=cat_colors.get(cat, "#888"),
                                clip_on=False, zorder=10))

for s in ax.spines.values():
    s.set_color("#999"); s.set_linewidth(0.6)

cb = fig.colorbar(im, cax=cax)
cb.set_label("Z-score vs national mean\n(red = above national, blue = below)", fontsize=9.5)
cb.ax.tick_params(labelsize=9)
cb.outline.set_linewidth(0.5); cb.outline.set_edgecolor("#999")

fig.suptitle("Community archetype fingerprints — Nigeria 37 states\n"
              "HAC k = 5, Ward linkage · Cell colour = z-score vs national mean · Cell text = raw cluster mean",
              fontsize=13, fontweight="bold", y=1.04, color="#1F3864")

legend_handles = [mpatches.Patch(color=v, label=k) for k, v in cat_colors.items()]
fig.legend(handles=legend_handles, loc="lower center", ncol=5,
           fontsize=9, frameon=True, framealpha=0.95, edgecolor="#CCC",
           bbox_to_anchor=(0.5, -0.04),
           title="Variable category", title_fontsize=9.5)

plt.tight_layout(rect=[0, 0.04, 1, 0.88])
plt.savefig(f"{OUTPUT_DIR}/fig_d5_archetype_fingerprint.png", dpi=300,
            bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: fig_d5_archetype_fingerprint.png")


### 7.4b State-to-archetype assignment table

A clean visual cross-walk of which states belong to which cluster. The companion to the fingerprint heatmap above.


In [23]:
# Figure: archetype state-assignment columns
import textwrap

fig, ax = plt.subplots(figsize=(15, 7.5), facecolor="white")
ax.set_facecolor(BG); ax.axis("off")

arch_palette = [ARCHETYPE_COLORS_NAMED[a] for a in archetypes_ordered]
n_cols = len(archetypes_ordered)
col_w  = 1.0 / n_cols * 0.96
gap    = (1.0 - col_w * n_cols) / (n_cols + 1)

for i, (arch, color) in enumerate(zip(archetypes_ordered, arch_palette)):
    x = gap + i * (col_w + gap)
    # Header strip (taller to accommodate wrapped descriptor)
    ax.add_patch(plt.Rectangle((x, 0.80), col_w, 0.15,
                                transform=ax.transAxes,
                                facecolor=color, edgecolor="white",
                                lw=1, zorder=2))
    parts = arch.split(" — ", 1)
    ax.text(x + col_w/2, 0.925, parts[0],
            transform=ax.transAxes, ha="center", va="center",
            fontsize=11, fontweight="bold", color="white")
    if len(parts) > 1:
        wrapped = "\n".join(textwrap.wrap(parts[1], width=24))
        ax.text(x + col_w/2, 0.855, wrapped,
                transform=ax.transAxes, ha="center", va="center",
                fontsize=7.5, color="white", style="italic")
    # State list (sorted by zero-dose rate within cluster)
    states = mod2.loc[mod2["archetype"]==arch].sort_values("zero_dose_2024", ascending=False)
    for j, (_, r) in enumerate(states.iterrows()):
        y = 0.73 - j * 0.040
        ax.text(x + 0.005, y, r["state_name"],
                transform=ax.transAxes, ha="left", va="top",
                fontsize=8.5, fontweight="bold")
        ax.text(x + col_w - 0.005, y, f"  ZD = {r['zero_dose_2024']:.0f}%",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=7.5, color="#555")
    # n badge
    ax.text(x + col_w/2, 0.02,
            f"n = {(mod2['archetype']==arch).sum()}",
            transform=ax.transAxes, ha="center", va="bottom",
            fontsize=10, fontweight="bold", color=color)

ax.set_title("State assignments to community archetypes\n"
              "Nigeria 37 states · HAC k = 5 · Ward linkage",
              fontsize=12, fontweight="bold", color="#1F3864", pad=10)

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig_d5_archetype_state_list.png", dpi=300,
            bbox_inches="tight", facecolor="white")
plt.show()
print("Saved: fig_d5_archetype_state_list.png")


Saved: fig_d5_archetype_state_list.png


### 7.5 PCA biplot

In [24]:
from sklearn.decomposition import PCA
pca  = PCA(n_components=2, random_state=42)
X_pc = pca.fit_transform(X_sc2)
var_exp = pca.explained_variance_ratio_*100

fig, ax = plt.subplots(figsize=(13, 9), facecolor=BG)
for arch in archetypes_ordered:
    idx_a = mod2["archetype"]==arch
    ax.scatter(X_pc[idx_a.values,0], X_pc[idx_a.values,1],
               c=ARCHETYPE_COLORS_NAMED[arch], s=120, alpha=0.85,
               edgecolors="white", linewidths=0.8, label=arch, zorder=4)
for i, state in enumerate(state_labels):
    ax.annotate(state, (X_pc[i,0]+0.06, X_pc[i,1]+0.06), fontsize=7.5, alpha=0.85, color="#333")

loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
for fi in np.argsort(np.abs(loadings[:,0]))[-5:]:
    ax.annotate("", xy=(loadings[fi,0]*3, loadings[fi,1]*3), xytext=(0,0),
                arrowprops=dict(arrowstyle="->", color="#555", lw=1.4))
    ax.text(loadings[fi,0]*3.2, loadings[fi,1]*3.2,
            cluster_features[fi].replace("pct_","").replace("_"," ").title(),
            fontsize=7.5, color="#555", ha="center")
ax.set_xlabel(f"PC1 ({var_exp[0]:.1f}% variance)", fontsize=10)
ax.set_ylabel(f"PC2 ({var_exp[1]:.1f}% variance)", fontsize=10)
ax.set_title("Domain 5-II — PCA Biplot of Community Archetypes", fontsize=12, fontweight="bold")
ax.legend(fontsize=9); ax.grid(alpha=0.15, ls="--")
ax.axhline(0, color="#bbb", lw=0.8); ax.axvline(0, color="#bbb", lw=0.8)
ax.set_facecolor(BG)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig_d5_pca_biplot.png", dpi=150, bbox_inches="tight")
plt.show()


---
## Section 8 — Save Outputs and Download (Colab)

In [25]:
out_cols = ["state_name","zone_name","cluster_id","archetype","zero_dose_2024"] + cluster_features
mod2[out_cols].to_csv(f"{OUTPUT_DIR}/d5_community_archetypes.csv", index=False)
print("✅ d5_community_archetypes.csv saved")

print("\nArchetype membership:")
for arch in archetypes_ordered:
    states = mod2[mod2["archetype"]==arch]["state_name"].tolist()
    zd_mean = mod2[mod2["archetype"]==arch]["zero_dose_2024"].mean()
    print(f"  {arch:<38}  n={len(states):2d}  mean ZD={zd_mean:.1f}%  →  {states}")


✅ d5_community_archetypes.csv saved

Archetype membership:
  Cluster A — Highest Burden, Weak ANC Coverage, Distance Barrier  n= 2  mean ZD=85.2%  →  ['Kebbi', 'Sokoto']
  Cluster B — High Burden, Distance Barrier, Rural Majority  n= 5  mean ZD=45.8%  →  ['Adamawa', 'Kogi', 'Kwara', 'Niger', 'Plateau']
  Cluster C — Moderate Burden, High Fertility, High Stunting  n=10  mean ZD=42.8%  →  ['Jigawa', 'Kaduna', 'Kano', 'Katsina', 'Zamfara', 'Bauchi', 'Borno', 'Gombe', 'Taraba', 'Yobe']
  Cluster D — Lower Burden, High Education, High Poverty  n= 5  mean ZD=18.0%  →  ['Benue', 'Nasarawa', 'Ebonyi', 'Akwa Ibom', 'Bayelsa']
  Cluster E — Lowest Burden, Urban Majority, High Facility Delivery  n=15  mean ZD=11.6%  →  ['FCT', 'Ekiti', 'Lagos', 'Ogun', 'Ondo', 'Osun', 'Oyo', 'Abia', 'Anambra', 'Enugu', 'Imo', 'Cross River', 'Delta', 'Edo', 'Rivers']


In [26]:
if IN_COLAB:
    from google.colab import files
    import os
    for f in sorted(os.listdir(OUTPUT_DIR)):
        if f.endswith((".png",".csv")):
            print(f"  Downloading {f}")
            files.download(f"{OUTPUT_DIR}/{f}")
    print("\n✅ All downloads complete.")
else:
    import os
    print("Local — outputs in:", OUTPUT_DIR)
    for f in sorted(os.listdir(OUTPUT_DIR)):
        if f.endswith((".png",".csv")):
            sz = os.path.getsize(f"{OUTPUT_DIR}/{f}")/1024
            print(f"  {sz:6.0f} KB  {f}")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All downloads complete.


In [ ]:
# ============================================================================
# [CIDRE-Quantium] CSV table-formatting pass  -  appended for GAVI review
# ----------------------------------------------------------------------------
# Purpose : make this notebook's GENERATED CSV tables presentation-ready.
#           (consistent rounding to 4 dp, integer count/rank columns, and a
#            styled .xlsx copy with a bold frozen header).
# Scope   : ONLY this notebook's own result tables, listed in _FMT_TABLES below.
#           Input/source data files are never touched.
# Safety  : runs AFTER all analysis cells; does not change any analysis or any
#           column names / row counts, so end-to-end behaviour is unchanged.
#           Every step is guarded - it can never raise and stop the notebook.
# ============================================================================
import os as _os
import numpy as _np
import pandas as _pd

try:
    _FMT_DIR = str(OUTPUT_DIR)            # this notebook's own output directory
except NameError:
    _FMT_DIR = "outputs"               # fallback if the cell is run in isolation

_FMT_TABLES = ["d1_national_at_risk_summary.csv", "d1_lga_at_risk_below_80pct.csv", "d1_lga_forecast_all.csv", "d5_community_archetypes.csv"]
_INT_HINTS  = ("count", "rank", "year", "yr", "pop", "cohort",
               "n_children", "susceptible", "_n")

def _fmt_one(_path):
    if not _os.path.exists(_path):
        return False
    try:
        _df = _pd.read_csv(_path)
    except Exception as _e:
        print("[format] skip", _os.path.basename(_path), "->", _e)
        return False
    for _col in _df.select_dtypes(include=[_np.number]).columns:
        _name = str(_col).lower(); _s = _df[_col]
        if any(_h in _name for _h in _INT_HINTS) and _s.dropna().mod(1).eq(0).all():
            _df[_col] = _s.astype("Int64")          # tidy whole-number counts
        else:
            _df[_col] = _s.round(4)                 # consistent decimals
    _df.to_csv(_path, index=False)                  # same schema, tidier values
    try:                                            # styled Excel companion
        from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
        from openpyxl.utils import get_column_letter
        _xp = _os.path.splitext(_path)[0] + ".xlsx"
        with _pd.ExcelWriter(_xp, engine="openpyxl") as _xw:
            _df.to_excel(_xw, index=False, sheet_name="Table")
            _ws = _xw.sheets["Table"]
            _fill = PatternFill("solid", fgColor="1F3B57")
            _font = Font(bold=True, color="FFFFFF")
            _thin = Side(style="thin", color="D9D9D9")
            _bd = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)
            for _j, _c in enumerate(_df.columns, 1):
                _cell = _ws.cell(row=1, column=_j)
                _cell.fill = _fill; _cell.font = _font
                _cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
                _ws.column_dimensions[get_column_letter(_j)].width = min(max(len(str(_c)) + 2, 10), 40)
            for _i in range(2, len(_df) + 2):
                for _j in range(1, len(_df.columns) + 1):
                    _ws.cell(row=_i, column=_j).border = _bd
            _ws.freeze_panes = "A2"; _ws.row_dimensions[1].height = 26
    except Exception as _e:
        print("[format] xlsx styling skipped for", _os.path.basename(_path), "->", _e)
    return True

_done = [_t for _t in _FMT_TABLES if _fmt_one(_os.path.join(_FMT_DIR, _t))]
if _FMT_TABLES:
    print("Table formatting complete -", len(_done), "of", len(_FMT_TABLES),
          "result table(s) formatted in '%s':" % _FMT_DIR)
    for _t in _done:
        print("   -", _t)
else:
    print("Domain produces figure/report outputs only - no result CSV tables to format.")


In [ ]:
# =====================================================================
# APPENDED (additive): map the HAC community archetypes back to the 37
# individual states and export clean, stakeholder-ready assignment tables.
# Reads the archetype assignments already written above; changes nothing else.
# =====================================================================
import pandas as _pd, os as _os
_ad = _pd.read_csv(_os.path.join(OUTPUT_DIR, "d5_community_archetypes.csv"))
_ad["Archetype"] = _ad["archetype"].str.extract(r"Cluster ([A-E])")
_ORDER = ["A", "B", "C", "D", "E"]
_DESC = {
    "A": "Highest burden; weak ANC; distance barrier",
    "B": "High burden; distance barrier; rural majority",
    "C": "Moderate burden; high fertility; high stunting",
    "D": "Lower burden; higher education; high poverty",
    "E": "Lowest burden; urban; high facility delivery",
}
_PKG = {
    "A": ("Intensive combined push: demand generation through trusted community and religious leaders "
          "and women's-education-sensitive messaging, paired with fixed-post and mobile outreach to "
          "overcome distance where ANC contact and facility delivery are near absent."),
    "B": ("Access-led: rural outreach and transport solutions to close the distance barrier and "
          "strengthen fixed posts, with moderate demand generation."),
    "C": ("High-volume routine scale-up sized to large, high-fertility birth cohorts; integrate with "
          "nutrition and antenatal contact; demand generation in high-fertility, Muslim-majority settings."),
    "D": ("Maintain and protect: poverty-sensitive measures to cut indirect costs, plus defaulter "
          "tracking; lighter, targeted support."),
    "E": ("Light maintenance: defaulter tracking and reminders that leverage high facility delivery "
          "and mobile-phone access."),
}
_ad["_o"] = _ad["Archetype"].map({c: i for i, c in enumerate(_ORDER)})

# (1) Tidy per-state assignment table (all 37 states)
_tidy = (_ad.sort_values(["_o", "zero_dose_2024"], ascending=[True, False])
            [["state_name", "zone_name", "Archetype", "zero_dose_2024"]]
            .rename(columns={"state_name": "State", "zone_name": "Zone",
                             "zero_dose_2024": "Zero_dose_2024_pct"}))
_tidy["Zero_dose_2024_pct"] = _tidy["Zero_dose_2024_pct"].round(1)
_tidy["Archetype_profile"] = _tidy["Archetype"].map(_DESC)
_tidy = _tidy[["State", "Zone", "Archetype", "Archetype_profile", "Zero_dose_2024_pct"]]
_tidy.to_csv(_os.path.join(OUTPUT_DIR, "d5_state_archetype_assignments.csv"), index=False)

# (2) Grouped archetype -> states -> intervention package table (5 rows)
_rows = []
for _c in _ORDER:
    _sub = _ad[_ad["Archetype"] == _c].sort_values("zero_dose_2024", ascending=False)
    _rows.append({
        "Archetype": _c,
        "Profile": _DESC[_c],
        "States_n": int(len(_sub)),
        "Mean_zero_dose_2024_pct": round(float(_sub["zero_dose_2024"].mean()), 1),
        "States": ", ".join(_sub["state_name"].tolist()),
        "Priority_intervention_package": _PKG[_c],
    })
_grp = _pd.DataFrame(_rows)
_grp.to_csv(_os.path.join(OUTPUT_DIR, "d5_archetype_intervention_packages.csv"), index=False)

print("Saved: d5_state_archetype_assignments.csv (37 states) and "
      "d5_archetype_intervention_packages.csv (5 archetypes)")
for _, _r in _grp.iterrows():
    print(f"  Cluster {_r['Archetype']} (n={_r['States_n']}, mean ZD {_r['Mean_zero_dose_2024_pct']}%): {_r['States']}")
